In [15]:
import numpy as np
import xml.etree.ElementTree as ET
from typing import List, Tuple, Dict
from xml.dom import minidom

def parse_polylines_by_image(xml_content: str) -> Dict[str, Dict]:
    """Parse polyline points from XML content, grouped by image name."""
    root = ET.fromstring(xml_content)
    image_polylines = {}
    
    for image in root.findall('.//image'):
        image_name = image.get('name')
        width = int(image.get('width'))
        height = int(image.get('height'))
        polylines = []
        
        for polyline in image.findall('.//polyline'):
            points_str = polyline.get('points')
            points = [tuple(map(float, point.split(','))) 
                     for point in points_str.split(';')]
            polylines.append(points)
        
        if polylines:
            image_polylines[image_name] = {
                'polylines': polylines,
                'width': width,
                'height': height
            }
    
    return image_polylines

def normalize_vector(v: np.ndarray) -> np.ndarray:
    """Normalize a vector."""
    norm = np.linalg.norm(v)
    if norm == 0:
        return v
    return v / norm

def polyline_to_polygon(points: List[Tuple[float, float]], width: float = 1.0) -> List[Tuple[float, float]]:
    """Convert a polyline to a polygon by giving it width."""
    if len(points) < 2:
        return []
    
    # Convert points to numpy array for easier calculation
    points = np.array(points)
    
    # Calculate vectors between consecutive points
    vectors = points[1:] - points[:-1]
    
    # Calculate normalized perpendicular vectors
    perp_vectors = np.zeros_like(vectors)
    perp_vectors[:, 0] = -vectors[:, 1]
    perp_vectors[:, 1] = vectors[:, 0]
    perp_vectors = np.array([normalize_vector(v) for v in perp_vectors])
    
    # Calculate the offset for each segment
    half_width = width / 2
    offsets = perp_vectors * half_width
    
    # Create the polygon points
    upper_points = []
    lower_points = []
    
    # Handle first point
    upper_points.append(tuple(points[0] + offsets[0]))
    lower_points.append(tuple(points[0] - offsets[0]))
    
    # Handle middle points
    for i in range(1, len(points) - 1):
        # Average the offset vectors for smooth transitions
        avg_offset = normalize_vector(offsets[i-1] + offsets[i]) * half_width
        upper_points.append(tuple(points[i] + avg_offset))
        lower_points.append(tuple(points[i] - avg_offset))
    
    # Handle last point
    upper_points.append(tuple(points[-1] + offsets[-1]))
    lower_points.append(tuple(points[-1] - offsets[-1]))
    
    # Combine points to form polygon (go up one side and down the other)
    polygon_points = upper_points + lower_points[::-1]
    
    return polygon_points

def create_xml_output(image_polygons: Dict[str, List[List[Tuple[float, float]]]]) -> str:
    """Create XML output with polygon annotations."""
    root = ET.Element('annotations')
    
    for image_name, data in image_polygons.items():
        image_elem = ET.SubElement(root, 'image')
        image_elem.set('name', image_name)
        image_elem.set('width', str(data['width']))
        image_elem.set('height', str(data['height']))
        
        for polygon_points in data['polygons']:
            polygon = ET.SubElement(image_elem, 'polygon')
            polygon.set('label', 'breaker')
            polygon.set('source', 'auto')
            polygon.set('occluded', '0')
            
            # Convert points to string format
            points_str = ';'.join(f'{x:.2f},{y:.2f}' for x, y in polygon_points)
            polygon.set('points', points_str)
            
            polygon.set('z_order', '0')
    
    # Pretty print the XML
    xml_str = minidom.parseString(ET.tostring(root)).toprettyxml(indent='  ')
    return xml_str

def main(xml_content: str, line_width: float = 1.0) -> str:
    """Main function to process XML and create polygon annotations."""
    # Parse original polylines
    image_data = parse_polylines_by_image(xml_content)
    
    # Process each image
    result_data = {}
    for image_name, data in image_data.items():
        # Convert each polyline to a polygon
        polygons = []
        for polyline in data['polylines']:
            polygon = polyline_to_polygon(polyline, width=line_width)
            if polygon:
                polygons.append(polygon)
        
        if len(polygons) > 0:
            # Store results
            result_data[image_name] = {
                'polygons': polygons,
                'width': data['width'],
                'height': data['height']
            }
        
        print(f"Processed {image_name}:")
        print(f"  Original polylines: {len(data['polylines'])}")
        print(f"  Generated polygons: {len(polygons)}")
    
    # Generate XML output
    return create_xml_output(result_data)

# Read input XML
with open('additional_data/annotations.xml', 'r') as f:
    xml_content = f.read()

# Process and generate new XML with thin polygons
output_xml = main(xml_content, line_width=1.0)

# Save output XML
with open('additional_data/annotations_poly.xml', 'w') as f:
    f.write(output_xml)

print("\nCreated annotations_poly.xml")

Processed 20211014T120100Z_t01553-02596_a1097.png:
  Original polylines: 22
  Generated polygons: 22
Processed 20211014T120100Z_t01553-02596_a1098.png:
  Original polylines: 20
  Generated polygons: 20
Processed 20211014T120100Z_t01553-02596_a1099.png:
  Original polylines: 19
  Generated polygons: 19
Processed 20211014T120100Z_t01553-02596_a1115.png:
  Original polylines: 26
  Generated polygons: 26
Processed 20211014T120100Z_t01553-02596_a1116.png:
  Original polylines: 26
  Generated polygons: 26
Processed 20211014T120100Z_t01553-02596_a1117.png:
  Original polylines: 28
  Generated polygons: 28
Processed 20211014T120100Z_t01553-02596_a1206.png:
  Original polylines: 26
  Generated polygons: 26
Processed 20211014T120100Z_t01553-02596_a1207.png:
  Original polylines: 27
  Generated polygons: 27
Processed 20211014T120100Z_t01553-02596_a1208.png:
  Original polylines: 27
  Generated polygons: 27
Processed 20211014T120100Z_t01553-02596_a1330.png:
  Original polylines: 33
  Generated pol

In [16]:
import numpy as np
import cv2
import xml.etree.ElementTree as ET
from typing import List, Tuple, Dict
import distinctipy

def parse_polygons_from_xml(xml_file: str) -> Dict[str, Dict]:
    """Parse polygon points from XML file."""
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    image_polygons = {}
    
    for image in root.findall('.//image'):
        image_name = image.get('name')
        width = int(image.get('width'))
        height = int(image.get('height'))
        
        polygons = []
        for polygon in image.findall('.//polygon'):
            points_str = polygon.get('points')
            points = [tuple(map(float, point.split(','))) 
                     for point in points_str.split(';')]
            polygons.append(points)
            
        if polygons:
            image_polygons[image_name] = {
                'polygons': polygons,
                'width': width,
                'height': height
            }
    
    return image_polygons

def create_colored_mask(polygons: List[List[Tuple[float, float]]], 
                       width: int, 
                       height: int) -> np.ndarray:
    """Create a colored mask where each polygon has a distinct color."""
    # Create empty RGB mask
    mask = np.zeros((height, width, 3), dtype=np.uint8)
    
    # Get distinct colors for each polygon
    n_colors = len(polygons)
    colors = distinctipy.get_colors(n_colors)
    
    # Convert colors to BGR (OpenCV format) and scale to 0-255
    colors_bgr = [(int(b * 255), int(g * 255), int(r * 255)) 
                  for r, g, b in colors]
    
    # Draw each polygon with its distinct color
    for polygon, color in zip(polygons, colors_bgr):
        # Convert points to integer array
        points = np.array(polygon, dtype=np.int32)
        
        # Draw filled polygon
        cv2.fillPoly(mask, [points], color)
    
    return mask

def main(xml_file: str, output_dir: str = 'visualization') -> None:
    """Main function to create colored visualizations of polygons."""
    import os
    os.makedirs(output_dir, exist_ok=True)
    
    # Parse polygons from XML
    image_data = parse_polygons_from_xml(xml_file)
    
    print(f"Found {len(image_data)} images with polygons")
    
    # Process each image
    for image_name, data in image_data.items():
        # Create colored mask
        mask = create_colored_mask(
            data['polygons'], 
            data['width'], 
            data['height']
        )
        
        # Save visualization
        output_name = f"viz_{image_name}"
        output_path = os.path.join(output_dir, output_name)
        cv2.imwrite(output_path, mask)
        
        print(f"\nProcessed {image_name}:")
        print(f"  Number of polygons: {len(data['polygons'])}")
        print(f"  Output saved as: {output_name}")

# Generate visualizations
main('additional_data/annotations_poly.xml', 'additional_data/visualization')

Found 110 images with polygons

Processed 20211014T120100Z_t01553-02596_a1097.png:
  Number of polygons: 22
  Output saved as: viz_20211014T120100Z_t01553-02596_a1097.png

Processed 20211014T120100Z_t01553-02596_a1098.png:
  Number of polygons: 20
  Output saved as: viz_20211014T120100Z_t01553-02596_a1098.png

Processed 20211014T120100Z_t01553-02596_a1099.png:
  Number of polygons: 19
  Output saved as: viz_20211014T120100Z_t01553-02596_a1099.png

Processed 20211014T120100Z_t01553-02596_a1115.png:
  Number of polygons: 26
  Output saved as: viz_20211014T120100Z_t01553-02596_a1115.png

Processed 20211014T120100Z_t01553-02596_a1116.png:
  Number of polygons: 26
  Output saved as: viz_20211014T120100Z_t01553-02596_a1116.png

Processed 20211014T120100Z_t01553-02596_a1117.png:
  Number of polygons: 28
  Output saved as: viz_20211014T120100Z_t01553-02596_a1117.png

Processed 20211014T120100Z_t01553-02596_a1206.png:
  Number of polygons: 26
  Output saved as: viz_20211014T120100Z_t01553-02596

In [19]:
import os, re

def pad_numbers(filename):
    # Pattern to match the numbers after Z_t and _a
    pattern = r'Z_t(\d+)-(\d+)_a(\d+)'
    
    # Replace function to pad each number group to 4 digits
    def pad_match(match):
        t1 = match.group(1).zfill(4)
        t2 = match.group(2).zfill(4)
        a = match.group(3).zfill(4)
        return f'Z_t{t1}-{t2}_a{a}'
    
    # Apply the replacement
    return re.sub(pattern, pad_match, filename)

def rename_files(directory='.', dry_run=False):
    changes_found = False
    for filename in os.listdir(directory):
        if filename.endswith('.png'):  # Only process PNG files
            new_name = pad_numbers(filename)
            if new_name != filename:
                changes_found = True
                if dry_run:
                    print(f'Would rename: {filename} → {new_name}')
                else:
                    os.rename(
                        os.path.join(directory, filename),
                        os.path.join(directory, new_name)
                    )
                    print(f'Renamed: {filename} → {new_name}')
    
    if not changes_found:
        print("No files found that need renaming.")
        
        
def pad_numbers_in_xml(xml_file):
    # Read the XML file
    with open(xml_file, 'r') as f:
        content = f.read()
    
    # Pattern to match the filenames and capture groups for numbers
    pattern = r'(20\d{6}T\d{6}Z)_t(\d+)-(\d+)_a(\d+)\.png'
    
    # Replace function to pad each number group to 4 digits
    def pad_match(match):
        prefix = match.group(1)
        t1 = match.group(2).zfill(4)
        t2 = match.group(3).zfill(4)
        a = match.group(4).zfill(4)
        return f'{prefix}_t{t1}-{t2}_a{a}.png'
    
    # Apply the replacement
    new_content = re.sub(pattern, pad_match, content)
    
    # Check if any changes were made
    if new_content != content:
        # Write to a new file with '_updated' suffix
        output_file = xml_file.rsplit('.', 1)[0] + '_updated.xml'
        with open(output_file, 'w') as f:
            f.write(new_content)
        print(f'Updated XML saved to: {output_file}')
    else:
        print('No changes were needed in the XML file.')

        
pad_numbers_in_xml('triplets_fully_labeled/annotations_poly.xml')        
rename_files('triplets_fully_labeled/images', dry_run=False)

Updated XML saved to: triplets_fully_labeled/annotations_poly_updated.xml
No files found that need renaming.
